In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import os
import omicverse as ov
import numpy as np
from scipy.sparse import csr_matrix

# 示例sparse matrix，你可以替换成自己的sparse matrix
data = np.array([1, 2, 3, 4, 5, 6])
row = np.array([0, 0, 1, 2, 2, 2])
col = np.array([0, 2, 2, 0, 1, 2])
sparse_matrix = csr_matrix((data, (row, col)), shape=(3, 3))

# 打印原始sparse matrix
print("原始sparse matrix:")
print(sparse_matrix.toarray())

# 定义函数，将sparse matrix中指定比例的非0数据置0
def set_zero(sparse_matrix, percentage):
    # 获取非0元素的索引
    non_zero_indices = sparse_matrix.nonzero()
    # 计算需要置0的非0元素数量
    num_to_zero = int(np.ceil(len(non_zero_indices[0]) * percentage / 100))
    # 随机选择需要置0的非0元素索引
    zero_indices = np.random.choice(len(non_zero_indices[0]), num_to_zero, replace=False)
    # 将选中的非0元素置0
    sparse_matrix[non_zero_indices[0][zero_indices], non_zero_indices[1][zero_indices]] = 0
    return sparse_matrix

# 分别将10%，20%，30%，40%，50%的非0数据置0，并打印结果
percentages = [10, 20, 30, 40, 50]
for percentage in percentages:
    modified_sparse_matrix = set_zero(sparse_matrix.copy(), percentage)
    print(f"\n将{percentage}%的非0数据置0后的sparse matrix:")
    print(modified_sparse_matrix.toarray())

In [ ]:
def set_zero(sparse_matrix, percentage):
    non_zero_indices = sparse_matrix.nonzero()
    num_to_zero = int(np.ceil(len(non_zero_indices[0]) * percentage / 100))
    zero_indices = np.random.choice(len(non_zero_indices[0]), num_to_zero, replace=False)
    sparse_matrix[non_zero_indices[0][zero_indices], non_zero_indices[1][zero_indices]] = 0
    return sparse_matrix



In [ ]:
# Set the directory for the datasets and the output directory
data_dir = 'Original_Simulated_Data'
output_dir = 'Processed_Simulated_Data/Simulated_Dropout_Data'
os.makedirs(output_dir, exist_ok=True)
percentages = [10, 20, 30, 40, 50]

for percentage in percentages:
    adata_rna = sc.read_h5ad('Original_Simulated_Data/Simulated_Dataset_1/SimulatedData_1_rna.h5ad')
    adata_atac = sc.read_h5ad('Original_Simulated_Data/Simulated_Dataset_1/SimulatedData_1_atac.h5ad')
    adata_rna = adata_rna.raw.to_adata()
    adata_rna.X = set_zero(adata_rna.X, percentage)
    adata_atac.X = set_zero(adata_atac.X, percentage)
    
    output_path = f'{output_dir}/Simulated_Dataset_1/rna_dropout_{percentage}.h5ad'
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    adata_rna.write_h5ad(output_path, compression='gzip')
    output_path = f'{output_dir}/Simulated_Dataset_1/atac_dropout_{percentage}.h5ad'
    adata_atac.write_h5ad(output_path, compression='gzip')

In [ ]:
percentages = [10, 20, 30, 40, 50]
for percentage in percentages:
    adata_rna = sc.read_h5ad('Original_Simulated_Data/Simulated_Dataset_1/SimulatedData_1_rna.h5ad')
    adata_rna = adata_rna.raw.to_adata()
    adata_rna.X = set_zero(adata_rna.X, percentage)
    adata_rna.obs['ground_truth'] = adata_rna.obs['cell_type']

    # Preprocess the RNA data
    sc.pp.highly_variable_genes(adata_rna, n_top_genes=3000)
    adata_rna = adata_rna[:, adata_rna.var['highly_variable'] == True]
    sc.tl.pca(adata_rna)
    sc.pp.neighbors(adata_rna)
    sc.tl.umap(adata_rna)

    # Perform clustering using PCA representation
    ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['X_pca'].shape[1],
                    use_rep='X_pca')
    ov.utils.cluster(adata_rna, use_rep='X_pca', method='leiden', resolution=0.6+percentage*0.005)

    # Plot the spatial clustering results
    sc.pl.spatial(adata_rna, color=['ground_truth', 'leiden'], spot_size=0.12, wspace=0.4)


In [ ]:
adata_rna.X